# State Estimation

State estimation (SE) determines the most likely operating state of a power system from a set of noisy, redundant measurements. While power flow computes the exact solution given precise inputs, state estimation works with real-world data that is inherently imprecise. It processes measurements such as bus voltage magnitudes, power injections, and branch flows, and produces the best estimate of all bus voltage magnitudes and angles.

ANDES provides two estimation algorithms. The weighted least squares (WLS) method minimizes the weighted sum of squared measurement residuals and is optimal under Gaussian noise. The least absolute value (LAV) method minimizes the weighted sum of absolute residuals and is robust to gross measurement errors. Redundancy in measurements provides both noise rejection and the ability to detect bad data through statistical tests.

This tutorial covers running state estimation in ANDES, constructing custom measurement sets, interpreting results, performing bad data detection with the chi-squared test, comparing WLS and LAV under bad data conditions, and handling multi-island systems.

In [1]:
# Reduce logging verbosity for PDF builds
import os
if os.environ.get('SPHINX_BUILD_PDF'):
    import andes
    _orig_config_logger = andes.config_logger
    def _quiet_logger(stream_level=20, *args, **kwargs):
        stream_level = max(stream_level, 30)
        return _orig_config_logger(stream_level, *args, **kwargs)
    andes.config_logger = _quiet_logger

## Setup

We begin by importing ANDES, NumPy, and the measurement framework. State estimation requires a converged power flow as the starting point, so we load a case and run power flow first.

In [2]:
%matplotlib inline

import numpy as np
import andes
from andes.se import Measurements

andes.config_logger(stream_level=20)

In [3]:
ss = andes.load(andes.get_case('ieee14/ieee14.json'))
ss.PFlow.run()

Working directory: "/home/hcui9/repos/andes/docs/source/tutorials"
> Loaded generated Python code in "/home/hcui9/.andes/pycode".
Parsing input file "/home/hcui9/repos/andes/andes/cases/ieee14/ieee14.json"...
Input file parsed in 0.0016 seconds.
Connectivity check completed in 0.0000 seconds.
-> System connectivity check results:
  No islanded bus detected.
  System is interconnected.
  Each island has a slack bus correctly defined and enabled.
System internal structure set up in 0.0118 seconds.

-> Power flow calculation
           Numba: Off
   Sparse solver: KLU
 Solution method: NR method
Power flow initialized in 0.0028 seconds.
0: |F(x)| = 0.5605182134
1: |F(x)| = 0.006202200332
2: |F(x)| = 5.819382827e-06
3: |F(x)| = 6.957087684e-12
Converged in 4 iterations in 0.0032 seconds.
Report saved to "ieee14_out.txt" in 0.0010 seconds.


True

## Running State Estimation

The simplest way to run state estimation is to call `ss.SE.run()` with no arguments. ANDES will automatically generate a default set of measurements from the power flow solution, add Gaussian noise, and estimate the system state using the WLS algorithm.

In [4]:
ss.SE.run()


-> State Estimation
          Method: WLS
       Tolerance: 0.0001
  Max iterations: 100
No measurements provided; using default set (all bus voltages + all bus injections, 42 measurements).
Adding angle reference at bus 1 for island with 14 buses.
WLS converged in 3 iterations, J = 30.5907
SE converged in 3 iterations in 0.0037 seconds.

-> SE Report
       Converged: True
      Iterations: 3
     Objective J: 30.5907
    Measurements: 43
          States: 28
      Redundancy: 1.54x
        Max |dV|: 0.00755324 pu
        Max |da|: 0.00745156 rad


True

The SE report shows convergence status, the number of iterations, the objective function value $J$, the number of measurements and states, and the measurement redundancy ratio. The estimation error (Max |dV| and Max |da|) compares the estimated state against the power flow solution.

After convergence, the estimated bus voltage magnitudes and angles are available through the `v_est` and `a_est` properties.

In [5]:
print(f"Converged: {ss.SE.converged}")
print(f"Estimated voltages (pu): {np.round(ss.SE.v_est, 4)}")
print(f"Estimated angles (rad):  {np.round(ss.SE.a_est, 4)}")

Converged: True
Estimated voltages (pu): [1.0239 1.0259 1.0078 1.0082 1.0128 1.0315 1.0232 1.0292 1.0247 1.0195
 1.0265 1.0179 1.0166 1.0239]
Estimated angles (rad):  [ 0.     -0.032  -0.0618 -0.0781 -0.0679 -0.1111 -0.0912 -0.0343 -0.1318
 -0.1338 -0.1279 -0.1245 -0.1342 -0.1676]


## Custom Measurements

In practice, not every bus and branch is instrumented. The `Measurements` class allows you to construct a custom measurement set that reflects the actual placement of meters in the system. Each measurement is characterized by its type, location, and standard deviation (sigma), which represents the expected measurement noise.

Four types of measurements are supported:

| Method | Measurement type | Description |
|--------|-----------------|-------------|
| `add_bus_voltage()` | Voltage magnitude | RTU or PMU voltage measurements |
| `add_bus_angle()` | Voltage angle | PMU angle measurements |
| `add_bus_injection()` | P and Q injection | Bus power injection measurements |
| `add_line_flow()` | P and Q flow | Branch power flow measurements (from-end) |

The standard deviation sigma controls the weight of each measurement in the WLS objective. A smaller sigma means the estimator trusts that measurement more. Typical values are 0.005-0.02 for voltage measurements and 0.02-0.05 for power measurements.

In [6]:
m = Measurements(ss)

# Voltage meters at all buses (high accuracy)
m.add_bus_voltage(sigma=0.01)

# Power injection meters at all buses
m.add_bus_injection(sigma_p=0.02, sigma_q=0.03)

# Generate noisy measurement values from the PFlow solution
m.generate_from_pflow(seed=42)

print(f"Total measurements: {m.nm}")
print(f"State dimension: {2 * ss.Bus.n}")
print(f"Redundancy: {m.nm / (2 * ss.Bus.n):.1f}x")

Total measurements: 42
State dimension: 28
Redundancy: 1.5x


The redundancy ratio (measurements divided by states) indicates how much extra information is available. A ratio above 1.0 means the system is overdetermined, which is necessary for state estimation to work. Higher redundancy provides better noise rejection and more robust bad data detection. A redundancy of 2-3x is typical for real systems.

Pass the custom measurements to `SE.run()` to use them instead of the default set.

In [7]:
ss.SE.run(measurements=m)


-> State Estimation
          Method: WLS
       Tolerance: 0.0001
  Max iterations: 100
Adding angle reference at bus 1 for island with 14 buses.
WLS converged in 2 iterations, J = 13.027
SE converged in 2 iterations in 0.0033 seconds.

-> SE Report
       Converged: True
      Iterations: 2
     Objective J: 13.027
    Measurements: 43
          States: 28
      Redundancy: 1.54x
        Max |dV|: 0.0059106 pu
        Max |da|: 0.00930009 rad


True

### Selective Meter Placement

You can also place measurements on specific buses or lines rather than the entire system. This is useful for modeling realistic SCADA configurations where only key substations have full instrumentation. The `bus_idx` and `line_idx` parameters accept a list of device indices.

In [8]:
m2 = Measurements(ss)

# Voltage at selected buses only
m2.add_bus_voltage(bus_idx=[1, 2, 3, 6, 8], sigma=0.005)

# PMU angle measurement at the slack bus
m2.add_bus_angle(bus_idx=[1], sigma=0.01)

# Injection meters at all buses
m2.add_bus_injection(sigma_p=0.02, sigma_q=0.03)

# Flow meters on selected lines
m2.add_line_flow(line_idx=['Line_1', 'Line_2', 'Line_3', 'Line_7', 'Line_8'],
                 sigma_p=0.02, sigma_q=0.03)

m2.generate_from_pflow(seed=42)

print(f"Total measurements: {m2.nm}")
ss.SE.run(measurements=m2)


-> State Estimation
          Method: WLS
       Tolerance: 0.0001
  Max iterations: 100
WLS converged in 2 iterations, J = 10.0405
SE converged in 2 iterations in 0.0057 seconds.

-> SE Report
       Converged: True
      Iterations: 2
     Objective J: 10.0405
    Measurements: 44
          States: 28
      Redundancy: 1.57x
        Max |dV|: 0.0082004 pu
        Max |da|: 0.0132314 rad


Total measurements: 44


True

### Low-Level `add()` Method

All convenience wrappers delegate to the `add()` method, which accepts a model name and variable name. This method is useful when you need fine-grained control over individual measurements.

In [9]:
m3 = Measurements(ss)

# Equivalent to m3.add_bus_voltage(bus_idx=[1, 2], sigma=0.005)
m3.add('Bus', 'v', idx=[1, 2], sigma=0.005)

# Equivalent to m3.add_bus_angle(bus_idx=[1], sigma=0.01)
m3.add('Bus', 'a', idx=[1], sigma=0.01)

print(f"Measurements added: {m3.nm}")

Measurements added: 3


## Zero-Noise Verification

A useful sanity check is to verify that state estimation recovers the exact power flow solution when given perfect (noise-free) measurements. In this case, the measurement vector is set to $z = h(x_{\text{true}})$ without any added noise. The WLS estimator should converge to the true state to within numerical precision.

This test validates the correctness of the measurement functions $h(x)$ and the Jacobian $H(x)$.

In [10]:
from andes.se import StaticEvaluator

m_exact = Measurements(ss)
m_exact.add_bus_voltage(sigma=0.01)
m_exact.add_bus_angle(sigma=0.01)
m_exact.add_bus_injection(sigma_p=0.02, sigma_q=0.03)
m_exact.add_line_flow(sigma_p=0.02, sigma_q=0.03)

# Set z = h(x_true) exactly (no noise)
m_exact.finalize()
ev = StaticEvaluator(ss, m_exact)
theta_true = np.array(ss.Bus.a.v, dtype=float)
Vm_true = np.array(ss.Bus.v.v, dtype=float)
m_exact.z = ev.h(theta_true, Vm_true)

ss.SE.run(measurements=m_exact)

v_err = np.max(np.abs(ss.SE.v_est - Vm_true))
a_err = np.max(np.abs(ss.SE.a_est - theta_true))
print(f"Max voltage error: {v_err:.2e} pu")
print(f"Max angle error:   {a_err:.2e} rad")


-> State Estimation
          Method: WLS
       Tolerance: 0.0001
  Max iterations: 100
WLS converged in 1 iterations, J = 0
SE converged in 1 iterations in 0.0017 seconds.

-> SE Report
       Converged: True
      Iterations: 1
     Objective J: 0
    Measurements: 96
          States: 28
      Redundancy: 3.43x
        Max |dV|: 0 pu
        Max |da|: 0 rad


Max voltage error: 0.00e+00 pu
Max angle error:   0.00e+00 rad


The errors are at the level of machine precision ($\sim 10^{-11}$), confirming that the measurement model and solver are implemented correctly.

## Bad Data Detection

The chi-squared ($\chi^2$) test is a standard method for detecting the presence of bad data in the measurement set. Under the assumption that measurement errors are Gaussian, the weighted sum of squared residuals $J(\hat{x}) = [z - h(\hat{x})]^T W [z - h(\hat{x})]$ follows a chi-squared distribution with $m - n$ degrees of freedom, where $m$ is the number of measurements and $n$ is the number of states.

If $J$ exceeds the chi-squared threshold at the desired confidence level, the test rejects the null hypothesis that all measurements are normal, indicating possible bad data.

In [11]:
# Run SE with normal measurements
m_normal = Measurements(ss)
m_normal.add_bus_voltage(sigma=0.01)
m_normal.add_bus_injection(sigma_p=0.02, sigma_q=0.03)
m_normal.generate_from_pflow(seed=42)

ss.SE.run(measurements=m_normal)

passed, J, threshold, dof = ss.SE.chi_squared_test()
print(f"Chi-squared test: {'PASSED' if passed else 'FAILED'}")
print(f"  J = {J:.4f}, threshold = {threshold:.4f}, dof = {dof}")


-> State Estimation
          Method: WLS
       Tolerance: 0.0001
  Max iterations: 100
Adding angle reference at bus 1 for island with 14 buses.
WLS converged in 2 iterations, J = 13.027
SE converged in 2 iterations in 0.0029 seconds.

-> SE Report
       Converged: True
      Iterations: 2
     Objective J: 13.027
    Measurements: 43
          States: 28
      Redundancy: 1.54x
        Max |dV|: 0.0059106 pu
        Max |da|: 0.00930009 rad


Chi-squared test: PASSED
  J = 13.0270, threshold = 24.9958, dof = 15


Now we inject a gross error into one measurement (a 10-sigma outlier) and observe the effect on the chi-squared test.

In [12]:
# Inject a gross error
m_bad = Measurements(ss)
m_bad.add_bus_voltage(sigma=0.01)
m_bad.add_bus_injection(sigma_p=0.02, sigma_q=0.03)
m_bad.generate_from_pflow(seed=42)

# Corrupt one voltage measurement with a 10-sigma outlier
m_bad.z[0] += 10 * m_bad.sigma[0]

ss.SE.run(measurements=m_bad)

passed, J, threshold, dof = ss.SE.chi_squared_test()
print(f"Chi-squared test: {'PASSED' if passed else 'FAILED'}")
print(f"  J = {J:.4f}, threshold = {threshold:.4f}, dof = {dof}")


-> State Estimation
          Method: WLS
       Tolerance: 0.0001
  Max iterations: 100
Adding angle reference at bus 1 for island with 14 buses.
WLS converged in 3 iterations, J = 109.02
SE converged in 3 iterations in 0.0037 seconds.

-> SE Report
       Converged: True
      Iterations: 3
     Objective J: 109.02
    Measurements: 43
          States: 28
      Redundancy: 1.54x
        Max |dV|: 0.0148127 pu
        Max |da|: 0.0046609 rad


Chi-squared test: FAILED
  J = 109.0202, threshold = 24.9958, dof = 15


The chi-squared test correctly detects the presence of bad data. In practice, after detection, further analysis (such as the largest normalized residual test) would be used to identify and remove the specific bad measurement.

## Robust Estimation with LAV

The weighted least squares estimator is optimal under Gaussian noise but is sensitive to gross measurement errors. The least absolute value (LAV) estimator provides a robust alternative by minimizing the weighted sum of absolute residuals rather than squared residuals:

$$\min_x \sum_i \frac{|z_i - h_i(x)|}{\sigma_i}$$

This L1-norm formulation assigns less influence to measurements with large residuals compared to the L2-norm used by WLS. The LAV problem is solved via iteratively reweighted least squares (IRLS), where each iteration solves a WLS subproblem with weights inversely proportional to the current residual magnitudes. Because IRLS converges more slowly than Gauss-Newton, a higher iteration limit is generally recommended.

The `lav` algorithm function can be passed to `SE.run()` through the `algorithm` parameter. The following example compares WLS and LAV on the corrupted measurement set from the previous section.

In [13]:
from andes.se import lav

# True state from the power flow solution
Vm_true = np.array(ss.Bus.v.v, dtype=float)
theta_true = np.array(ss.Bus.a.v, dtype=float)

# WLS errors (from the run in the previous section on m_bad)
wls_v_err = np.max(np.abs(ss.SE.v_est - Vm_true))
wls_a_err = np.max(np.abs(ss.SE.a_est - theta_true))

# Run LAV on the same corrupted measurements
ss.SE.run(measurements=m_bad, algorithm=lav)
lav_v_err = np.max(np.abs(ss.SE.v_est - Vm_true))
lav_a_err = np.max(np.abs(ss.SE.a_est - theta_true))

print(f"{'Metric':<20} {'WLS':>12} {'LAV':>12}")
print(f"{'':-<44}")
print(f"{'Max |dV| (pu)':<20} {wls_v_err:>12.6f} {lav_v_err:>12.6f}")
print(f"{'Max |da| (rad)':<20} {wls_a_err:>12.6f} {lav_a_err:>12.6f}")


-> State Estimation
          Method: WLS
       Tolerance: 0.0001
  Max iterations: 100
LAV converged in 34 iterations, J = 27.1812
SE converged in 34 iterations in 0.0161 seconds.

-> SE Report
       Converged: True
      Iterations: 34
     Objective J: 27.1812
    Measurements: 43
          States: 28
      Redundancy: 1.54x
        Max |dV|: 0.00442327 pu
        Max |da|: 0.00523227 rad


Metric                        WLS          LAV
--------------------------------------------
Max |dV| (pu)            0.014813     0.004423
Max |da| (rad)           0.004661     0.005232


The LAV estimator produces smaller estimation errors than WLS in the presence of gross measurement errors. The L1-norm formulation effectively downweights the corrupted measurement, preventing it from biasing the state estimate. This robustness comes at the cost of slower convergence, as the IRLS procedure typically requires more iterations than the standard Gauss-Newton method.

Note that the chi-squared test is defined for the WLS objective function and is not applicable after an LAV run. ANDES will issue a warning if `chi_squared_test()` is called following LAV estimation.

## Multi-Island Systems

In systems with multiple electrical islands (disconnected components), each island requires its own voltage angle reference because absolute angles are not observable from power measurements alone. ANDES handles this automatically by detecting islands through connectivity analysis and adding a pseudo-measurement for the angle reference at the slack bus of each island.

The Kundur two-area system provides a convenient example when configured as two separate islands.

In [14]:
ss_island = andes.load(andes.get_case('kundur/kundur_islands.xlsx'),
                       default_config=True)
ss_island.PFlow.run()

print(f"Number of islands: {len(ss_island.Bus.island_sets)}")

Working directory: "/home/hcui9/repos/andes/docs/source/tutorials"
> Reloaded generated Python code of module "pycode".
Parsing input file "/home/hcui9/repos/andes/andes/cases/kundur/kundur_islands.xlsx"...


Input file parsed in 0.1925 seconds.
Connectivity check completed in 0.0000 seconds.
-> System connectivity check results:
  No islanded bus detected.
  System contains 2 island(s).
  Each island has a slack bus correctly defined and enabled.
System internal structure set up in 0.0116 seconds.

-> Power flow calculation
           Numba: Off
   Sparse solver: KLU
 Solution method: NR method
Power flow initialized in 0.0023 seconds.
0: |F(x)| = 14.9282832
1: |F(x)| = 3.670045871
2: |F(x)| = 0.1542941496
3: |F(x)| = 0.0003701017904
4: |F(x)| = 2.966633161e-09
Converged in 5 iterations in 0.0045 seconds.
Report saved to "kundur_islands_out.txt" in 0.0006 seconds.


Number of islands: 2


In [15]:
m_island = Measurements(ss_island)
m_island.add_bus_voltage(sigma=0.01)
m_island.add_bus_injection(sigma_p=0.02, sigma_q=0.03)
m_island.generate_from_pflow(seed=42)

ss_island.SE.run(measurements=m_island)


-> State Estimation
          Method: WLS
       Tolerance: 0.0001
  Max iterations: 100
Adding angle reference at bus 1 for island with 5 buses.
Adding angle reference at bus 3 for island with 5 buses.
WLS converged in 2 iterations, J = 8.07042
SE converged in 2 iterations in 0.0035 seconds.

-> SE Report
       Converged: True
      Iterations: 2
     Objective J: 8.07042
    Measurements: 32
          States: 20
      Redundancy: 1.60x
        Max |dV|: 0.00799312 pu
        Max |da|: 0.00501073 rad


True

The log shows that ANDES added angle references for both islands automatically. Without these references, the WLS gain matrix would be singular and the estimator would fail.

## Configuration

The SE routine provides several configuration options accessible through `ss.SE.config`.

In [16]:
ss.SE.config

OrderedDict([('sparselib', 'klu'),
             ('linsolve', 0),
             ('tol', 0.0001),
             ('max_iter', 100),
             ('flat_start', 0),
             ('report', 1)])

| Option | Default | Description |
|--------|---------|-------------|
| `tol` | 1e-4 | Convergence tolerance for the SE iteration |
| `max_iter` | 100 | Maximum number of iterations |
| `flat_start` | 0 | Use flat start (1.0 pu, 0 rad) instead of PFlow solution |
| `report` | 1 | Print result summary after estimation |

The flat start option is useful for testing robustness of convergence when the power flow solution is not available as an initial guess.

In [17]:
# Run with flat start
ss.SE.config.flat_start = 1
ss.SE.run(measurements=m_normal)

# Reset to default
ss.SE.config.flat_start = 0


-> State Estimation
          Method: WLS
       Tolerance: 0.0001
  Max iterations: 100
WLS converged in 3 iterations, J = 13.027
SE converged in 3 iterations in 0.0031 seconds.

-> SE Report
       Converged: True
      Iterations: 3
     Objective J: 13.027
    Measurements: 43
          States: 28
      Redundancy: 1.54x
        Max |dV|: 0.00591063 pu
        Max |da|: 0.00930006 rad


## Cleanup

In [18]:
!andes misc -C

"/home/hcui9/repos/andes/docs/source/tutorials/kundur_islands_out.txt" removed.
"/home/hcui9/repos/andes/docs/source/tutorials/ieee14_out.txt" removed.


## Next Steps

- {doc}`07-eigenvalue-analysis` - Small-signal stability assessment
- {doc}`09-contingency-analysis` - N-1 contingency screening